# Attention Mechanism Explained

Step-by-step implementation of the Transformer attention mechanism using NumPy.

**Topics:**
- Self-attention from scratch
- Multi-head attention
- Positional encoding
- Mini transformer block

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use("seaborn-v0_8-whitegrid")

## 1. Self-Attention Step by Step

Self-attention allows each token to "attend" to all other tokens.

Given input $ (sequence of embeddings):
1. Compute Query, Key, Value:  = XW_Q$,  = XW_K$,  = XW_V$
2. Attention scores: $	ext{scores} = rac{QK^T}{\sqrt{d_k}}$
3. Attention weights: $lpha = 	ext{softmax}(	ext{scores})$
4. Output: $	ext{out} = lpha V$

In [ ]:
def softmax(x, axis=-1):
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / e_x.sum(axis=axis, keepdims=True)

# Example: 4 tokens, embedding dim = 8
np.random.seed(42)
seq_len = 4
d_model = 8
d_k = d_v = 6  # key/value dimension

# Token embeddings (simulating a sentence)
tokens = ["The", "cat", "sat", "down"]
X = np.random.randn(seq_len, d_model)  # [4, 8]
print(f"Input X shape: {X.shape} (seq_len={seq_len}, d_model={d_model})")

In [ ]:
# Step 1: Project to Q, K, V
W_Q = np.random.randn(d_model, d_k) * 0.1
W_K = np.random.randn(d_model, d_k) * 0.1
W_V = np.random.randn(d_model, d_v) * 0.1

Q = X @ W_Q  # [4, 6]
K = X @ W_K  # [4, 6]
V = X @ W_V  # [4, 6]

print(f"Q shape: {Q.shape}")
print(f"K shape: {K.shape}")
print(f"V shape: {V.shape}")

# Step 2: Attention scores
scores = (Q @ K.T) / np.sqrt(d_k)  # [4, 4]
print(f"
Attention scores (before softmax):
{scores.round(3)}")

In [ ]:
# Step 3: Softmax to get attention weights
attn_weights = softmax(scores)  # [4, 4]

# Step 4: Weighted sum of values
output = attn_weights @ V  # [4, 6]

print(f"Attention weights (each row sums to 1):
{attn_weights.round(3)}")
print(f"
Row sums: {attn_weights.sum(axis=1).round(3)}")
print(f"
Output shape: {output.shape}")

In [ ]:
# Visualize attention weights
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(attn_weights, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(seq_len)); ax.set_xticklabels(tokens)
ax.set_yticks(range(seq_len)); ax.set_yticklabels(tokens)
ax.set_xlabel("Key (attending to)"); ax.set_ylabel("Query (from)")
ax.set_title("Self-Attention Weights")
for i in range(seq_len):
    for j in range(seq_len):
        ax.text(j, i, f"{attn_weights[i,j]:.2f}", ha="center", va="center", fontsize=10)
plt.colorbar(im)
plt.tight_layout()
plt.show()

## 2. Masked Self-Attention (for Decoders)

In autoregressive generation, tokens can only attend to previous tokens (causal mask).

In [ ]:
# Causal mask: upper triangle = -inf
mask = np.triu(np.ones((seq_len, seq_len)), k=1) * (-1e9)
masked_scores = scores + mask
masked_attn = softmax(masked_scores)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, weights, title in zip(axes, [attn_weights, masked_attn], ["Bidirectional (Encoder)", "Causal/Masked (Decoder)"]):
    im = ax.imshow(weights, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(seq_len)); ax.set_xticklabels(tokens)
    ax.set_yticks(range(seq_len)); ax.set_yticklabels(tokens)
    ax.set_title(title)
    for i in range(seq_len):
        for j in range(seq_len):
            ax.text(j, i, f"{weights[i,j]:.2f}", ha="center", va="center", fontsize=9)
plt.tight_layout()
plt.show()

## 3. Multi-Head Attention

Multiple attention heads let the model attend to different aspects simultaneously.

49248	ext{MultiHead}(Q,K,V) = 	ext{Concat}(	ext{head}_1, ..., 	ext{head}_h)W^O49248

In [ ]:
class MultiHeadAttention:
    def __init__(self, d_model, n_heads):
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.d_model = d_model
        
        # Initialize projections
        scale = 0.1
        self.W_Q = np.random.randn(d_model, d_model) * scale
        self.W_K = np.random.randn(d_model, d_model) * scale
        self.W_V = np.random.randn(d_model, d_model) * scale
        self.W_O = np.random.randn(d_model, d_model) * scale
    
    def split_heads(self, x):
        """Reshape (seq_len, d_model) -> (n_heads, seq_len, d_k)"""
        seq_len = x.shape[0]
        x = x.reshape(seq_len, self.n_heads, self.d_k)
        return x.transpose(1, 0, 2)  # (n_heads, seq_len, d_k)
    
    def forward(self, X, mask=None):
        Q = self.split_heads(X @ self.W_Q)
        K = self.split_heads(X @ self.W_K)
        V = self.split_heads(X @ self.W_V)
        
        # Scaled dot-product attention per head
        scores = (Q @ K.transpose(0, 2, 1)) / np.sqrt(self.d_k)
        if mask is not None:
            scores += mask
        
        self.attn_weights = softmax(scores)  # (n_heads, seq_len, seq_len)
        context = self.attn_weights @ V  # (n_heads, seq_len, d_k)
        
        # Concatenate heads
        context = context.transpose(1, 0, 2).reshape(-1, self.d_model)
        
        return context @ self.W_O

# Test
mha = MultiHeadAttention(d_model=8, n_heads=2)
output = mha.forward(X)
print(f"Input shape: {X.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {mha.attn_weights.shape} (n_heads, seq_len, seq_len)")

In [ ]:
# Visualize attention per head
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for h, ax in enumerate(axes):
    im = ax.imshow(mha.attn_weights[h], cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(seq_len)); ax.set_xticklabels(tokens)
    ax.set_yticks(range(seq_len)); ax.set_yticklabels(tokens)
    ax.set_title(f"Head {h+1}")
    for i in range(seq_len):
        for j in range(seq_len):
            ax.text(j, i, f"{mha.attn_weights[h,i,j]:.2f}", ha="center", va="center", fontsize=9)
plt.suptitle("Multi-Head Attention - Different Heads Learn Different Patterns")
plt.tight_layout()
plt.show()

## 4. Positional Encoding

Since attention has no notion of order, we add positional information:

49248PE_{(pos, 2i)} = \sin(pos / 10000^{2i/d_{model}})49248
49248PE_{(pos, 2i+1)} = \cos(pos / 10000^{2i/d_{model}})49248

In [ ]:
def positional_encoding(max_len, d_model):
    pe = np.zeros((max_len, d_model))
    position = np.arange(max_len).reshape(-1, 1)
    div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))
    
    pe[:, 0::2] = np.sin(position * div_term)
    pe[:, 1::2] = np.cos(position * div_term)
    return pe

# Visualize
pe = positional_encoding(50, 64)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
im = ax1.imshow(pe, aspect="auto", cmap="RdBu")
ax1.set_xlabel("Embedding Dimension"); ax1.set_ylabel("Position")
ax1.set_title("Positional Encoding Matrix")
plt.colorbar(im, ax=ax1)

# Show individual dimensions
for dim in [0, 1, 4, 5, 20, 21]:
    ax2.plot(pe[:, dim], label=f"dim {dim}")
ax2.set_xlabel("Position"); ax2.set_ylabel("Value")
ax2.set_title("Individual PE Dimensions")
ax2.legend(ncol=2)
plt.tight_layout()
plt.show()

## 5. Mini Transformer Block

A transformer block = Multi-Head Attention + Feed-Forward + Layer Norm + Residual connections

In [ ]:
def layer_norm(x, eps=1e-6):
    mean = x.mean(axis=-1, keepdims=True)
    std = x.std(axis=-1, keepdims=True)
    return (x - mean) / (std + eps)

class TransformerBlock:
    def __init__(self, d_model, n_heads, d_ff):
        self.attn = MultiHeadAttention(d_model, n_heads)
        # Feed-forward network
        self.W1 = np.random.randn(d_model, d_ff) * 0.1
        self.b1 = np.zeros(d_ff)
        self.W2 = np.random.randn(d_ff, d_model) * 0.1
        self.b2 = np.zeros(d_model)
    
    def feed_forward(self, x):
        # ReLU activation in between
        hidden = np.maximum(0, x @ self.W1 + self.b1)
        return hidden @ self.W2 + self.b2
    
    def forward(self, x, mask=None):
        # Self-attention + residual + layer norm
        attn_out = self.attn.forward(x, mask)
        x = layer_norm(x + attn_out)
        
        # Feed-forward + residual + layer norm
        ff_out = self.feed_forward(x)
        x = layer_norm(x + ff_out)
        
        return x

# Test transformer block
block = TransformerBlock(d_model=8, n_heads=2, d_ff=32)
out = block.forward(X)
print(f"Transformer block input:  {X.shape}")
print(f"Transformer block output: {out.shape}")
print(f"
Output (layer-normed, so mean≈0, std≈1):")
print(f"  Mean per token: {out.mean(axis=1).round(4)}")
print(f"  Std per token:  {out.std(axis=1).round(4)}")

In [ ]:
# Stack multiple transformer blocks
class MiniTransformer:
    def __init__(self, d_model, n_heads, d_ff, n_layers, max_len=512):
        self.blocks = [TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)]
        self.pe = positional_encoding(max_len, d_model)
    
    def forward(self, x, mask=None):
        seq_len = x.shape[0]
        x = x + self.pe[:seq_len]  # Add positional encoding
        
        for block in self.blocks:
            x = block.forward(x, mask)
        return x

# Create a 3-layer mini transformer
transformer = MiniTransformer(d_model=8, n_heads=2, d_ff=32, n_layers=3)
final_out = transformer.forward(X)
print(f"3-layer Transformer output shape: {final_out.shape}")
print(f"
This is what gets fed to task-specific heads (classification, generation, etc.)")

## Key Takeaways

| Component | Purpose |
|-----------|--------|
| Self-Attention | Let tokens communicate with each other |
| Multi-Head | Attend to different relationship types |
| Positional Encoding | Inject sequence order information |
| Residual + LayerNorm | Stable deep training |
| Feed-Forward | Per-token non-linear transformation |

**The Transformer revolution:**
- GPT: Decoder-only (causal mask) for generation
- BERT: Encoder-only (bidirectional) for understanding
- T5: Encoder-Decoder for seq2seq tasks

**Next:** Implement a character-level language model with this architecture!